#IMPORTS

In [ ]:
import os
import numpy as np
import torch 
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from matplotlib import pyplot as plt
import glob
import random
import torch.nn as nn
from torchsummary import summary

#Data_Loader

In [ ]:
import os
from torchvision.io import read_image
import numpy as np

class CustomImageDataset(Dataset):
    def __init__(self, pat_name, img_dir,label_dir):
        self.img_names = pat_name
        self.img_dir = img_dir
        self.label_dir = label_dir

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        #print(img_path)
        label_path = os.path.join(self.label_dir, self.img_names[idx])
        image = np.load(img_path)
        image = image.astype('float32')
        image = np.transpose(image,(3,2,0,1))
        image= torch.from_numpy(image)
        image = image[1,:,:,:]
        image = image.unsqueeze(0)
        #image = image.type(torch.LongTensor)
        #image=image.to('cuda')
        #image=transform(image)
        label = np.load(label_path)
        label = label.astype('float32') 
        label = np.transpose(label,(2,0,1))
        label= torch.from_numpy(label)
        #label = label.type(torch.LongTensor)
        #label=label.to('cuda')
        #label=transform(label)
        return image, label

#Directory

In [ ]:
img_path="../fdg/train_image/"
label_path="../fdg/train_label/"
img_list=os.listdir(img_path)
training_data=CustomImageDataset(img_list,img_path,label_path)

#Testing_dataloader_batchsize=2

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=2, shuffle=True)

In [ ]:
dataiter = iter(train_dataloader)
images, labels = dataiter.__next__()
print(type(images))
print(images.shape)
print(labels.shape)
print(torch.unique(labels[0,:,:,:]))
print(torch.unique(labels[1,:,:,:]))

#Visualize

In [ ]:
#batch = random.randint(0,1)
batch=1
#n_slice = random.randint(arr_0[0],arr_0[-1])
n_slice = random.randint(0,15)
a = images[batch,:,n_slice,:,:]
# a=a.cpu()
print(n_slice)
plt.figure(figsize=(12, 8))
plt.subplot(221)
plt.imshow(a[0])
plt.title('Channel 1')
# plt.subplot(222)
# plt.imshow(a[1])
# plt.title('Channel 2')
plt.subplot(222)
plt.imshow(labels[batch,n_slice,:,:])
plt.title('Mask')
plt.show()

In [ ]:
#MODEL 0

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_op=nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Conv3d(out_channels, out_channels, kernel_size=(3, 3, 3), padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True))
    def forward(self, x):
        return self.conv_op(x)
class DownSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool3d((2,2,2))
        #changed depth index 0 from 1 to 2 to reduce the size of the image
    def forward(self, x):
        down = self.conv(x)
        p = self.pool(down)
        return down, p
class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, in_channels//2, kernel_size=(2,2,2), stride=(2,2,2))
        self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1=self.up(x1)
        x=torch.cat([x1,x2],1)
        return self.conv(x)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        c=8
        self.down_convolution_1 = DownSample(in_channels, c)
        self.down_convolution_2 = DownSample(c, 2*c)
        self.down_convolution_3 = DownSample(2*c, 4*c)
        self.down_convolution_4 = DownSample(4*c, 8*c)

        self.bottle_neck = DoubleConv(8*c,16*c)

        self.up_convolution_1 = UpSample(16*c,8*c)
        self.up_convolution_2 = UpSample(8*c,4*c)
        self.up_convolution_3 = UpSample(4*c,2*c)
        self.up_convolution_4 = UpSample(2*c,c)

        self.logit= nn.Conv3d(in_channels=c, out_channels=num_classes, kernel_size=(1,1,1))
        self.out = nn.Sigmoid()
    def forward(self, x):
        # print("x.shape")
        # print(x.shape)
        down_1, p1 = self.down_convolution_1(x)
        # print("down_1.shape,p1.shape")
        # print(down_1.shape,p1.shape)
        down_2, p2 = self.down_convolution_2(p1)
        # print("down_2.shape,p2.shape")
        # print(down_2.shape,p2.shape)
        down_3, p3 = self.down_convolution_3(p2)
        # print("down_3.shape,p3.shape")
        # print(down_3.shape,p3.shape)
        down_4, p4 = self.down_convolution_4(p3)
        # print("down_4.shape,p4.shape")
        # print(down_4.shape,p4.shape)
        b = self.bottle_neck(p4)
        # print("b.shape")
        # print(b.shape)
        up_1 = self.up_convolution_1(b, down_4)
        # print("up_1.shape")
        # print(up_1.shape)
        up_2 = self.up_convolution_2(up_1, down_3)
        # print("up_2.shape")
        # print(up_2.shape)
        up_3 = self.up_convolution_3(up_2, down_2)
        # print("up_3.shape")
        # print(up_3.shape)
        up_4 = self.up_convolution_4(up_3, down_1)
        # print("up_4.shape")
        # print(up_4.shape)
        logit = self.logit(up_4)
        out = self.out(logit)

        return out

In [ ]:
#MODEL SUMMARY

In [ ]:
model= UNet(1,1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = model.to(device)
#input_T = torch.randn(2,2,662,400,400)
summary(model, input_size=(1, 16, 400, 400))

In [ ]:
img_path="../fdg/train_image/"
label_path="../fdg/train_label/"
img_list=os.listdir(img_path)
training_data=CustomImageDataset(img_list,img_path,label_path)
val_img_path="../fdg/val_image/"
val_label_path="../fdg/val_label/"
val_img_list=os.listdir(val_img_path)
val_data=CustomImageDataset(val_img_list,val_img_path,val_label_path)

In [ ]:
train_dataloader = DataLoader(training_data, batch_size=2, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=2, shuffle=True)
dataiter = iter(train_dataloader)
val_dataiter = iter(val_dataloader)
images, labels = dataiter.__next__()
val_images, val_labels = val_dataiter.__next__()
#print(val_labels.shape)

In [ ]:
#LOSS FUNCTION

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-15):
        super().__init__()
        self.eps = eps
        
    def forward(self, y_pred, y_true):

        dice_coef = 0.
        
        intersection = y_true[:].float() * y_pred[:].float()
        union = y_true[:].float() + y_pred[:].float()
        
        dice_coef += ((2*intersection.sum() + self.eps)/(union.sum()+self.eps))
        return 1.-dice_coef 

In [ ]:
def apply_threshold_iou(output, threshold=1):
    return torch.where(output >= threshold, 1, output)

In [ ]:
#IOU

In [ ]:
class IoU(nn.Module):
    def __init__(self, eps=1e-15):
        super().__init__()
        self.eps = eps
        
    def forward(self, y_pred, y_true):

        iou = 0.
        y_true=y_true.detach()
        y_pred=y_pred.detach()
        
        intersection = y_true[:].float() * y_pred[:].float()
        union = y_true[:].float() + y_pred[:].float()
        union = apply_threshold_iou(union)
        iou += ((intersection.sum() + self.eps)/(union.sum()+self.eps))
        return iou

In [ ]:
import torch.optim as optim

criterion = DiceLoss()
jaccard=IoU()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

MENTIONING THE CSV FILE PATH OF CSV FILE

In [ ]:
import csv
file_path = '../fdg/stats/channel_1_c8.csv'

In [ ]:

data = [['Epoch', 'Train_Loss', 'Val_Loss', 'IoU']]

with open(file_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(data)

# with open(file_path, 'a', newline='') as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerows(new_data)

In [ ]:
def apply_threshold(output, threshold):
    return torch.where(output >= threshold, 1, 0)

In [ ]:
#EARLY STOPPER

In [ ]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
#TRAINING

In [ ]:
#default
val_history = 1.0
iou_history = 0.0
save_epoch = 0

In [ ]:
early_stopper = EarlyStopper(patience=10, min_delta=0)


for epoch in range(0,100):
    print(f"EPOCH\n{epoch}\n********************************")
    running_loss = 0.0
    total_val_loss = 0.0
    total_iou = 0.0
    training_loss = 0.0

    # Training phase
    model.train()
    for i, data in enumerate(train_dataloader, 0):
        inputs, labels = data
        inputs = inputs.cuda(non_blocking=True)
        labels = labels.cuda(non_blocking=True)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        outputs = torch.squeeze(outputs)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        training_loss += loss.item()

        # Cleanup
        del inputs, labels, outputs, loss
        torch.cuda.empty_cache()
        print(i)
        if i % 100 == 99:
            print(f"RUNNING LOSS\n[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}")
            running_loss = 0.0

    avg_train_loss = training_loss / len(train_dataloader)
    print(f"TRAINING LOSS\n[{epoch + 1}] training_loss: {avg_train_loss:.3f}")

    # Validation phase
    model.eval()
    total_val_loss = 0.0
    total_iou = 0.0

    with torch.no_grad():  # Critical: No gradients during validation
        for j, val_data in enumerate(val_dataloader):
            val_inputs, val_labels = val_data
            val_inputs = val_inputs.cuda(non_blocking=True)
            val_labels = val_labels.cuda(non_blocking=True)
            
            val_outputs = model(val_inputs)
            val_outputs = torch.squeeze(val_outputs)
            val_outputs = apply_threshold(val_outputs, 0.5)
            
            val_loss = criterion(val_outputs, val_labels)
            iou = jaccard(val_outputs, val_labels)
            
            total_val_loss += val_loss.item()
            total_iou += iou.item()

            # Cleanup
            del val_inputs, val_labels, val_outputs, val_loss, iou
            torch.cuda.empty_cache()

    avg_val_loss = total_val_loss / len(val_dataloader)
    avg_iou = total_iou / len(val_dataloader)
    
    print(f"IOU SCORE: {avg_iou:.3f}\nPREVIOUS: {iou_history:.3f}")
    print(f"VALIDATION LOSS: {avg_val_loss:.3f}\nPREVIOUS: {val_history:.3f}")

    # Save results to CSV
    new_data = [[epoch+1, avg_train_loss, avg_val_loss, avg_iou]]
    with open(file_path, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(new_data)
    
    # Early stopping check
    if early_stopper.early_stop(avg_val_loss):
        print("SATURATION POINT REACHED")
        break

    # Save best model
    if val_history >= avg_val_loss:
        iou_history = avg_iou
        val_history = avg_val_loss
        save_epoch = epoch + 1
        torch.save(model.state_dict(), "../fdg/weights/channel_1_c8.pt")
        print("WEIGHTS SAVED")

print('Finished Training')
print(save_epoch)

In [ ]:
torch.save(model.state_dict(), "../fdg/weights/channel_1_c16_try.pt")

In [ ]:
#LOAD BEST MODEL WEIGHTS

In [ ]:
model.load_state_dict(torch.load('../fdg/weights/channel_1_c16.pt'))
model.eval()

In [ ]:
val_img_path="./numpy/dataset/val/data/"
val_label_path="./numpy/dataset/val/label/"
val_img_list=os.listdir(val_img_path)
val_data=CustomImageDataset(val_img_list,val_img_path,val_label_path)

In [ ]:
val_dataloader = DataLoader(val_data, batch_size=2, shuffle=True)
val_dataiter = iter(val_dataloader)
val_images, val_labels = val_dataiter.__next__()
print(torch.unique(val_labels[0,:,:,:]))
print(torch.unique(val_labels[1,:,:,:]))

In [ ]:
val_images=val_images.cuda()
val_output=model(val_images)
val_output=torch.squeeze(val_output)
val_threshold_output= apply_threshold(val_output,0.5)
val_output=val_output.cpu()
val_threshold_output=val_threshold_output.cpu()
val_output=val_output.detach()
val_images=val_images.cpu()

In [ ]:
channel=1

In [ ]:
#VISUALIZE PREDICTED IMAGES

In [ ]:
slide=12
plt.figure(figsize=(12, 8))
plt.subplot(231)
plt.imshow(val_images[channel,0,slide,:,:])
plt.title('Channel 1')
plt.subplot(232)
plt.imshow(val_images[channel,1,slide,:,:])
plt.title('Channel 2')
plt.subplot(233)
plt.imshow(val_labels[channel,slide,:,:])
plt.title('Label')
plt.subplot(234)
plt.imshow(val_output[channel,slide,:,:])
plt.title('Prediction')
plt.subplot(235)
plt.imshow(val_threshold_output[channel,slide,:,:])
plt.title('Threshold Prediction')

In [ ]:
torch.unique(val_output[1,:,:,:])

In [ ]:
torch.unique(val_threshold_output[1,:,:,:])